In [1]:
import sys
import subprocess
from importlib.metadata import version

requirements = [
    "qiskit==2.2.3",
    "numpy>=1.26,<3",
    "scipy>=1.11,<2",
    "matplotlib>=3.8,<4",
    "packaging>=23",
    "scikit-learn>=1.4,<2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True
)
print("Python:", sys.version)
print(
    "패키지:",
    {
        name: version(name)
        for name in ["qiskit", "numpy", "scipy", "matplotlib", "scikit-learn"]
    },
)
# 새 Python 프로세스에서 import와 설치된 직접 의존성을 확인합니다.
check = r"""
from importlib.metadata import version, requires
from packaging.requirements import Requirement
import qiskit, numpy, scipy, matplotlib
import sklearn

packages = ["qiskit", "numpy", "scipy", "matplotlib", "scikit-learn"]
for name in packages:
    for text in requires(name) or []:
        dependency = Requirement(text)
        if dependency.marker is None or dependency.marker.evaluate():
            actual = version(dependency.name)
            assert not dependency.specifier or dependency.specifier.contains(
                actual, prereleases=True
            ), (name, str(dependency), actual)
print("패키지 import 및 직접 의존성 확인: 통과")
"""
subprocess.run([sys.executable, "-c", check], check=True)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
패키지: {'qiskit': '2.2.3', 'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'scikit-learn': '1.6.1'}


CompletedProcess(args=['/usr/bin/python3', '-c', '\nfrom importlib.metadata import version, requires\nfrom packaging.requirements import Requirement\nimport qiskit, numpy, scipy, matplotlib\nimport sklearn\n\npackages = ["qiskit", "numpy", "scipy", "matplotlib", "scikit-learn"]\nfor name in packages:\n    for text in requires(name) or []:\n        dependency = Requirement(text)\n        if dependency.marker is None or dependency.marker.evaluate():\n            actual = version(dependency.name)\n            assert not dependency.specifier or dependency.specifier.contains(\n                actual, prereleases=True\n            ), (name, str(dependency), actual)\nprint("패키지 import 및 직접 의존성 확인: 통과")\n'], returncode=0)

In [2]:
import numpy as np

expectation = -0.6
score = (1 - expectation) / 2
label = 1
loss = -(label * np.log(score) + (1 - label) * np.log(1 - score))
print("점검 점수:", score, "BCE:", round(float(loss), 6))
assert np.isclose(score, 0.8)

점검 점수: 0.8 BCE: 0.223144


In [4]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


# theta(세타): Y축 회전각, 라디안
def f(theta):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    return float(
        Statevector.from_instruction(qc).probabilities() @ np.array([1.0, -1.0])
    )


theta = 0.4
step = 1e-6
shift = (f(theta + np.pi / 2) - f(theta - np.pi / 2)) / 2
finite = (f(theta + step) - f(theta - step)) / (2 * step)
print("파라미터 이동:", shift, "유한차분:", finite)
assert np.isclose(shift, finite, atol=1e-8)

파라미터 이동: -0.38941834230865047 유한차분: -0.3894183423502895


In [5]:
import time
from pathlib import Path
from importlib.metadata import version
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# 1. 실행 환경과 기본 설정
started = time.perf_counter()
print(
    "버전:",
    {k: version(k) for k in ["qiskit", "numpy", "scipy", "scikit-learn", "matplotlib"]},
)
# [정규화 진동, 정규화 온도, 상태]: 0 정상, 1 점검 필요

# 2. 학습 데이터와 시험 데이터
train = np.array(
    [
        [0.10, 0.15, 0],
        [0.15, 0.35, 0],
        [0.25, 0.20, 0],
        [0.30, 0.40, 0],
        [0.40, 0.25, 0],
        [0.45, 0.40, 0],
        [0.55, 0.65, 1],
        [0.60, 0.85, 1],
        [0.70, 0.60, 1],
        [0.75, 0.90, 1],
        [0.85, 0.70, 1],
        [0.90, 0.85, 1],
    ]
)
test = np.array(
    [
        [0.20, 0.30, 0],
        [0.35, 0.35, 0],
        [0.40, 0.45, 0],
        [0.60, 0.70, 1],
        [0.70, 0.80, 1],
        [0.80, 0.65, 1],
    ]
)
layers = 2
maxiter = 80
threshold = 0.5
X, y = train[:, :2], train[:, 2].astype(int)
Xt, yt = test[:, :2], test[:, 2].astype(int)
bits = (np.arange(16)[:, None] >> np.arange(4)) & 1
zavg = (1 - 2 * bits).mean(axis=1)


# 3. 양자 회로와 기대값 계산
def expectation(x, w):
    qc = QuantumCircuit(4)
    for q in range(4):
        qc.ry(np.pi * x[0], q)
        qc.rz(np.pi * x[1], q)
    for layer in range(layers):
        qc.cx(0, 1)
        qc.cx(1, 2)
        qc.cx(2, 3)
        for q in range(4):
            # theta(세타): 학습 가능한 회전각, 라디안
            k = 8 * layer + 2 * q
            qc.ry(w[k], q)
            qc.rz(w[k + 1], q)
    return float(Statevector.from_instruction(qc).probabilities() @ zavg)


# 4. 점검 점수와 학습 손실
def prob(x, w):
    return (1 - expectation(x, w)) / 2


def loss(w):
    v = np.clip([prob(x, w) for x in X], 1e-8, 1 - 1e-8)
    return float(-np.mean(y * np.log(v) + (1 - y) * np.log(1 - v)))


# 파라미터 이동은 기대값에 적용하고 손실은 연쇄법칙으로 미분합니다.

# 5. 초기 각도와 기대값 미분 확인
w0 = np.linspace(-0.3, 0.3, 8 * layers)
x0 = X[0]
wp = w0.copy()
wm = w0.copy()
wp[0] += np.pi / 2
wm[0] -= np.pi / 2
shift = (expectation(x0, wp) - expectation(x0, wm)) / 2
eps = 1e-6
wp = w0.copy()
wm = w0.copy()
wp[0] += eps
wm[0] -= eps
finite = (expectation(x0, wp) - expectation(x0, wm)) / (2 * eps)
assert np.isclose(shift, finite, atol=1e-7)

# 각도 탐색 중 목적함수 값을 기록합니다.
history = []


def objective(w):
    value = loss(w)
    history.append(value)
    return value


# 6. 각도 최적화와 시험 데이터 예측
result = minimize(objective, w0, method="COBYLA", options={"maxiter": maxiter})
pt = np.array([prob(x, result.x) for x in Xt])
pred = (pt >= threshold).astype(int)

# 7. 고전 기준 모델과 성능 비교
base = LogisticRegression(C=1.0, max_iter=500).fit(X, y)
bp = base.predict(Xt)
assert np.isfinite(pt).all() and ((pt >= 0) & (pt <= 1)).all()
print("학습/시험 크기:", X.shape, Xt.shape)
print("파라미터 이동/유한차분:", shift, finite)
print("초기/최종 BCE:", loss(w0), loss(result.x))
print("손실 평가 횟수:", result.nfev, "학습 회로 평가 횟수:", result.nfev * len(X))
print("최적화 상태:", result.success, str(result.message))
for name, pr in [("QNN", pred), ("LogisticRegression", bp)]:
    print(
        name,
        "정확도:",
        accuracy_score(yt, pr),
        "F1:",
        f1_score(yt, pr, zero_division=0),
    )
    print(
        "혼동행렬 [실제0,실제1] × [예측0,예측1]:",
        confusion_matrix(yt, pr, labels=[0, 1]).tolist(),
    )
print("시험 점검 점수:", np.round(pt, 6).tolist())
print("시험 정답/예측:", yt.tolist(), pred.tolist())

# 8. 학습 그래프와 예측 결과 저장
out = Path("day39_qml")
out.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 3), facecolor="white")
ax.plot(history, color="darkorange")
ax.set(xlabel="Loss evaluation", ylabel="Training BCE")
fig.tight_layout()
fig.savefig(out / "loss.png", dpi=150)
plt.close(fig)
np.savez(out / "results.npz", weights=result.x, test_score=pt, prediction=pred)
print("파일:", out / "loss.png", out / "results.npz")
print("시간(초):", round(time.perf_counter() - started, 3))

버전: {'qiskit': '2.2.3', 'numpy': '2.4.6', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
학습/시험 크기: (12, 2) (6, 2)
파라미터 이동/유한차분: 0.23555152367267723 0.2355515241192485
초기/최종 BCE: 0.5105833734978632 0.29215646518932764
손실 평가 횟수: 80 학습 회로 평가 횟수: 960
최적화 상태: False Return from COBYLA because the objective function has been evaluated MAXFUN times.
QNN 정확도: 1.0 F1: 1.0
혼동행렬 [실제0,실제1] × [예측0,예측1]: [[3, 0], [0, 3]]
LogisticRegression 정확도: 1.0 F1: 1.0
혼동행렬 [실제0,실제1] × [예측0,예측1]: [[3, 0], [0, 3]]
시험 점검 점수: [0.208007, 0.295263, 0.334758, 0.68536, 0.835352, 0.816216]
시험 정답/예측: [0, 0, 0, 1, 1, 1] [0, 0, 0, 1, 1, 1]
파일: day39_qml/loss.png day39_qml/results.npz
시간(초): 3.55


혼동행렬의 행은 실제 상태, 열은 예측 상태입니다. 왼쪽 위 3은 정상 세 건을 정상으로 분류한 수이고 오른쪽 아래 3은 점검 필요 세 건을 맞힌 수입니다. 학습 손실 감소는 학습 자료에 대한 적합 정도를 나타냅니다. 시험 여섯 건의 결과로 산업 전체의 예측 성능을 추정하기에는 자료가 제한적입니다. 두 모델의 정확도와 F1이 같으므로 양자 모델의 업무상 이득을 주장하려면 계산 비용과 추가 데이터의 성능을 평가해야 합니다.

손실 그래프의 가로축은 목적함수 평가 순서입니다. COBYLA가 탐색한 후보 손실은 중간에 증가할 수 있습니다. `loss.png`는 저장된 그래프이며 `results.npz`에는 학습 각도와 시험 점수·예측이 들어 있습니다.